# 01. EDA
DAY1 EDA 노트북. 질문(Q1~Q5)마다 **그래프 → 해석 → 시사점(모델링 반영)** 순서로 정리한다.
- 데이터 : `preprocess.py` 결과 (`processed/battery_data.pkl`)
- 분석 대상 : 제외 셀(`excluded=True`)을 뺀 Batch 1 (36셀), Batch 2 (39셀), Batch 3 (44셀)
- 그래프는 `results/figures/` 에 저장 (PDF 첨부용)

In [ ]:
import os, sys, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# 노트북 위치가 루트든 notebooks/ 든 preprocess.py를 찾도록 설정
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
from preprocess import run_all, EOL_CAPACITY

FIG_DIR = os.path.join(ROOT, 'results', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, f'{name}.png'), dpi=150, bbox_inches='tight')

plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})
BATCH_LABEL = {'b1': 'Batch 1 (train)', 'b2': 'Batch 2 (test)', 'b3': 'Batch 3 (extra)'}
BATCH_COLOR = {'b1': '#1f77b4', 'b2': '#ff7f0e', 'b3': '#2ca02c'}

In [ ]:
data = run_all()
cell_info_all = data['cell_info']
summary_all = data['summary']

ci = cell_info_all[~cell_info_all['excluded']].copy()           # 분석 대상 셀
ci['structure'] = np.where(ci['policy'].str.contains('newstructure'), 'new', 'old')   # 셀 구조 구분
summary = summary_all[summary_all['cell_id'].isin(ci['cell_id']) & ~summary_all['is_outlier']].copy()

print(ci.groupby('batch').size().rename('n_cells'))

---
## Q1. Cycle Life 분포는 어떻게 생겼는가?
- 배치별 분포 형태 (히스토그램, 박스플롯)
- 장수명(>1000) / 단수명(<500) 비율, 분류 기준(550) 기준 클래스 비율
- 배치 간 분포 차이 검정 (KS test)
- 이상치 셀 식별 : 왜 유독 짧은가?

### Q1-1. 배치별 기초 통계량

In [ ]:
def life_stats(s):
    return pd.Series({
        'n': len(s), 'mean': s.mean(), 'std': s.std(), 'CV(%)': s.std() / s.mean() * 100,
        'min': s.min(), 'Q1': s.quantile(.25), 'median': s.median(), 'Q3': s.quantile(.75), 'max': s.max(),
        'skew': stats.skew(s), 'skew(log)': stats.skew(np.log10(s)),
        'long(>1000) %': (s > 1000).mean() * 100,
        'short(<500) %': (s < 500).mean() * 100,
        'label=1(>=550) %': (s >= 550).mean() * 100,
    })

q1_stats = ci.groupby('batch')['cycle_life'].apply(life_stats).unstack().round(2)
q1_stats.loc['all'] = life_stats(ci['cycle_life']).round(2)
q1_stats

### Q1-2. 분포 시각화 (히스토그램 / 박스플롯)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [2, 1]})
bins = np.arange(0, ci['cycle_life'].max() + 100, 100)

for b, g in ci.groupby('batch'):
    axes[0].hist(g['cycle_life'], bins=bins, alpha=0.5, color=BATCH_COLOR[b],
                 edgecolor='white', label=f"{BATCH_LABEL[b]} (n={len(g)})")
    axes[0].axvline(g['cycle_life'].median(), color=BATCH_COLOR[b], ls='--', lw=1.5)
for x, txt in [(500, 'short<500'), (1000, 'long>1000')]:
    axes[0].axvline(x, color='gray', ls=':', lw=1)
    axes[0].text(x, axes[0].get_ylim()[1] * 0.95, txt, color='gray', ha='center', fontsize=9)
axes[0].set(xlabel='Cycle Life', ylabel='Number of cells', title='Cycle Life Histogram (dashed = median)')
axes[0].legend()

order = sorted(ci['batch'].unique())
axes[1].boxplot([ci.loc[ci.batch == b, 'cycle_life'] for b in order],
                tick_labels=[BATCH_LABEL[b].split(' (')[0] for b in order], widths=0.5)
rng = np.random.default_rng(0)
for i, b in enumerate(order, 1):
    y = ci.loc[ci.batch == b, 'cycle_life']
    axes[1].scatter(i + rng.uniform(-0.12, 0.12, len(y)), y, s=15, alpha=0.6, color=BATCH_COLOR[b])
axes[1].set(ylabel='Cycle Life', title='Box Plot by Batch')

plt.suptitle('Q1. Cycle Life Distribution', fontweight='bold')
plt.tight_layout(); savefig('Q1_cycle_life_distribution'); plt.show()

### Q1-3. 로그 변환 효과 (타깃 변환 검토)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for b, g in ci.groupby('batch'):
    axes[0].hist(g['cycle_life'], bins=20, alpha=0.5, color=BATCH_COLOR[b], label=BATCH_LABEL[b])
    axes[1].hist(np.log10(g['cycle_life']), bins=20, alpha=0.5, color=BATCH_COLOR[b], label=BATCH_LABEL[b])
axes[0].set(title=f"Raw (skew = {stats.skew(ci['cycle_life']):.2f})", xlabel='Cycle Life')
axes[1].set(title=f"log10 (skew = {stats.skew(np.log10(ci['cycle_life'])):.2f})", xlabel='log10(Cycle Life)')
axes[0].legend()
plt.suptitle('Q1. Target Transform: Raw vs log10', fontweight='bold')
plt.tight_layout(); savefig('Q1_log_transform'); plt.show()

### Q1-4. 배치 간 분포 차이 검정 (Kolmogorov-Smirnov)

In [ ]:
pairs = [('b1', 'b2'), ('b1', 'b3'), ('b2', 'b3')]
ks_rows = []
for a, b in pairs:
    x, y = ci.loc[ci.batch == a, 'cycle_life'], ci.loc[ci.batch == b, 'cycle_life']
    r = stats.ks_2samp(x, y)
    ks_rows.append({'pair': f'{a} vs {b}', 'KS stat': round(r.statistic, 3), 'p-value': round(r.pvalue, 4),
                    'median diff': x.median() - y.median(),
                    '판정(α=0.05)': '분포 다름' if r.pvalue < 0.05 else '차이 없음'})
pd.DataFrame(ks_rows)

### Q1-5. 이상치 셀 : 왜 유독 짧은가?
배치별 수명 하위 셀의 충전 정책(1단계 C-rate), 초기 100사이클 평균 충전시간·최고온도·용량을
배치 중앙값과 비교한다.

In [ ]:
def parse_policy(p):
    # '5.4C(40%)-3.6C' -> (5.4, 40, 3.6). 형식이 다르면 NaN
    m = re.match(r'([\d.]+)C\((\d+)%\)-([\d.]+)C', str(p))
    return pd.Series(dict(zip(['C1', 'switch_SOC', 'C2'], map(float, m.groups())))) if m else \
           pd.Series({'C1': np.nan, 'switch_SOC': np.nan, 'C2': np.nan})

early = (summary[(summary['cycle'] >= 2) & (summary['cycle'] <= 100)]
         .groupby('cell_id')[['chargetime', 'Tmax', 'Tavg', 'QD', 'IR']].mean()
         .add_prefix('early_'))
q1_cells = ci.merge(early, on='cell_id')
q1_cells = pd.concat([q1_cells, q1_cells['policy'].apply(parse_policy)], axis=1)

# 배치별 IQR 기준 이상치 (Q1 - 1.5*IQR 미만 / Q3 + 1.5*IQR 초과)
def iqr_flag(s):
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    return np.where(s < q1 - 1.5 * iqr, 'low', np.where(s > q3 + 1.5 * iqr, 'high', ''))
q1_cells['iqr_outlier'] = q1_cells.groupby('batch')['cycle_life'].transform(iqr_flag)

cols = ['batch', 'cell_id', 'policy', 'cycle_life', 'iqr_outlier', 'C1', 'early_chargetime', 'early_Tmax', 'early_QD']
shortest = q1_cells.sort_values('cycle_life').groupby('batch').head(3)[cols]
batch_median = q1_cells.groupby('batch')[['cycle_life', 'C1', 'early_chargetime', 'early_Tmax', 'early_QD']].median()

print('[배치별 수명 하위 3개 셀]')
display(shortest.round(3))
print('[배치 중앙값]')
display(batch_median.round(3))
print('[IQR 기준 이상치 셀]')
display(q1_cells[q1_cells['iqr_outlier'] != ''][cols].round(3))

In [ ]:
# 하위 셀 vs 나머지 : 초기 충전시간 / 최고온도와 수명 관계 (색 = 배치, 하위 3개 강조)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, col, lab in [(axes[0], 'early_chargetime', 'Mean charge time, cycles 2-100 (min)'),
                     (axes[1], 'early_Tmax', 'Mean Tmax, cycles 2-100 (C)')]:
    for b, g in q1_cells.groupby('batch'):
        ax.scatter(g[col], g['cycle_life'], color=BATCH_COLOR[b], alpha=0.6, label=BATCH_LABEL[b])
    s = q1_cells[q1_cells['cell_id'].isin(shortest['cell_id'])]
    ax.scatter(s[col], s['cycle_life'], facecolors='none', edgecolors='red', s=120, lw=1.5, label='shortest 3 / batch')
    ax.set(xlabel=lab, ylabel='Cycle Life')
axes[0].legend(fontsize=9)
plt.suptitle('Q1. Why are some cells short-lived?', fontweight='bold')
plt.tight_layout(); savefig('Q1_short_life_cells'); plt.show()

### Q1-6. 배치 효과 확인 : 셀 구조(newstructure) / 충전시간

In [ ]:
print('[배치 x 셀 구조별 수명]')
display(ci.groupby(['batch', 'structure'])['cycle_life'].describe()[['count', 'mean', 'min', '50%', 'max']].round(0))
print('[같은 정책, 다른 구조 : Batch 2]')
b2 = ci[ci.batch == 'b2'].assign(base_policy=lambda d: d.policy.str.replace('-newstructure', ''))
both = b2.groupby('base_policy')['structure'].nunique()
display(b2[b2.base_policy.isin(both[both > 1].index)]
        .sort_values(['base_policy', 'structure'])[['cell_id', 'policy', 'structure', 'cycle_life']])
print('[배치별 초기 충전시간 (cycles 2-100 평균)]')
display(q1_cells.groupby('batch')['early_chargetime'].describe()[['mean', 'std', 'min', 'max']].round(2))

### Q1 정리

**[핵심 발견]**
- 분포 형태 : Batch 1(학습) 534~1074 / 중앙값 772 / 대칭(skew 0.29),
  Batch 2(테스트) 392~1186 / 중앙값 472 / 72%가 500 미만(skew 1.57),
  Batch 3 541~1935 / 중앙값 1006 / 가장 장수명 (skew 1.21)
- 장/단수명 비율 : 550 기준 장수명 비율 Batch 1 97%, Batch 2 23%, Batch 3 98% → 학습 배치의 단수명 셀은 1개
- 배치 간 차이 : KS test 결과 모든 배치 쌍의 분포가 다름 (p < 0.001)
- 학습-테스트 범위 불일치 : Batch 2 대부분은 학습 최솟값(534) 미만, Batch 3 절반 이상은 학습 최댓값(1074) 초과 → 외삽 필요
- 셀 구조 효과 : Batch 1은 전부 기존 구조, Batch 3은 전부 newstructure, Batch 2는 혼재
  - Batch 2 기존 구조 30셀 : 평균 453 (392~514) → 전부 학습 최솟값 미만
  - Batch 2 newstructure 9셀 : 평균 942 (777~1186) → IQR 장수명 이상치 9개 전부 해당
  - 같은 정책(4.8C(80%)-4.8C)도 구조에 따라 수명 차이 → 정책(C-rate)만으로 수명 설명 불가
  - (주의) "newstructure"의 정확한 의미(셀/실험 조건 변경)는 데이터에서 확인 불가
- 충전시간 : Batch 1은 9.0~12.1분(std 0.81)으로 다양, Batch 2는 10.04~10.24분(std 0.06)으로 거의 일정 → 테스트 배치에서 구분 신호 없음
- 단수명 셀 : 충전량 대부분을 5C 이상 고전류로 채우는 정책 (5.4C(80%)-5.4C, 3.6C(9%)-5C, 3.7C(31%)-5.9C). 온도는 34.9~39.8℃로 일관된 패턴 없음

**[시사점 → 모델링 반영]**
- 태스크 : 회귀 선택 (분류는 학습 배치에 단수명 셀이 1개뿐이라 학습 불가)
- 모델 : 외삽 가능한 log 피처 기반 선형 모델(ElasticNet, Ridge) 우선, 트리 계열(RF, XGBoost)은 비교용
- 피처 : 정책·충전시간처럼 "실험 조건"을 나타내는 피처는 배치·구조가 바뀌면 일반화되지 않음
  → 셀의 실제 열화 상태를 반영하는 피처(ΔQ(V), 용량 변화) 중심으로 설계 (Q3에서 확인)
- 타깃 : log10(cycle_life) 검토 (전체 skew 1.04 → 0.03, MAPE와 정합). Q3에서 확정
- 평가 : Batch 2 결과를 구조별(old/new)로 나눠 오류 분석 → Gap(Valid-Test) 해석 근거

---
## Q2. 열화 곡선 : 방전 용량이 어떻게 감소하는가?
- 배치별 사이클-용량 곡선 (색 = 수명)
- 정규화 곡선 : 열화 속도가 일정한가, 가속되는가?
- 열화 속도 비교 : 초기(2~100) / 중기(수명 40~60%) / 말기(수명 마지막 10%)
- Knee point : 급격한 열화 시작점
- 초기 100사이클 확대 : 용량 값만으로 수명을 구분할 수 있는가?

In [ ]:
# 셀별 정상 행 + 평활화(rolling median 5) 용량
curves = (summary[['batch', 'cell_id', 'cycle', 'QD']].sort_values(['cell_id', 'cycle']).copy())
curves['QD_s'] = curves.groupby('cell_id')['QD'].transform(lambda s: s.rolling(5, center=True, min_periods=1).median())
curves = curves.merge(ci[['cell_id', 'cycle_life', 'structure']], on='cell_id')
curves = curves[curves['cycle'] <= curves['cycle_life']]                 # EOL 이후(Batch 2) 제외
curves['cycle_norm'] = curves['cycle'] / curves['cycle_life']
curves['QD_norm'] = curves['QD_s'] / curves.groupby('cell_id')['QD_s'].transform(lambda s: s.iloc[:10].median())
print(curves.groupby('batch')['cell_id'].nunique())

### Q2-1. 배치별 열화 곡선

In [ ]:
cmap = plt.get_cmap('viridis')
norm = plt.Normalize(ci['cycle_life'].min(), ci['cycle_life'].max())
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, (b, g) in zip(axes, curves.groupby('batch')):
    for cid, c in g.groupby('cell_id'):
        ax.plot(c['cycle'], c['QD_s'], color=cmap(norm(c['cycle_life'].iloc[0])), lw=0.8, alpha=0.8)
    ax.axhline(EOL_CAPACITY, color='red', ls='--', lw=1.2, label='EOL 0.88Ah')
    ax.set(title=BATCH_LABEL[b], xlabel='Cycle', ylim=(0.86, 1.11))
    ax.legend(loc='lower left')
axes[0].set_ylabel('Discharge capacity QD (Ah)')
fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, label='Cycle Life', pad=0.01)
plt.suptitle('Q2. Capacity Fade Curves (color = cycle life)', fontweight='bold')
savefig('Q2_capacity_fade'); plt.show()

### Q2-2. 정규화 곡선 : 열화가 선형인가, 가속되는가?

In [ ]:
grid = np.linspace(0, 1, 101)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, (b, g) in zip(axes, curves.groupby('batch')):
    interp = []
    for cid, c in g.groupby('cell_id'):
        y = np.interp(grid, c['cycle_norm'], c['QD_norm'])
        interp.append(y)
        ax.plot(grid, y, color='gray', lw=0.5, alpha=0.4)
    ax.plot(grid, np.median(interp, axis=0), color=BATCH_COLOR[b], lw=2.5, label='median')
    ax.plot([0, 1], [1, np.median(interp, axis=0)[-1]], 'k:', lw=1.2, label='linear reference')
    ax.set(title=BATCH_LABEL[b], xlabel='Normalized cycle (cycle / cycle life)')
    ax.legend(loc='lower left')
axes[0].set_ylabel('Normalized capacity (QD / initial QD)')
plt.suptitle('Q2. Normalized Fade Curves : linear vs accelerating', fontweight='bold')
plt.tight_layout(); savefig('Q2_normalized_fade'); plt.show()

### Q2-3. 열화 속도 : 초기 vs 말기 + Q2-4. Knee point

In [ ]:
def fade_features(c):
    c = c.sort_values('cycle')
    life = c['cycle_life'].iloc[0]
    early = c[(c['cycle'] >= 2) & (c['cycle'] <= 100)]
    mid = c[(c['cycle'] >= 0.4 * life) & (c['cycle'] <= 0.6 * life)]
    late = c[c['cycle'] >= 0.9 * life]
    slope_early = -np.polyfit(early['cycle'], early['QD_s'], 1)[0] * 100 if len(early) > 10 else np.nan   # Ah / 100 cycles
    slope_mid = -np.polyfit(mid['cycle'], mid['QD_s'], 1)[0] * 100 if len(mid) > 5 else np.nan
    slope_late = -np.polyfit(late['cycle'], late['QD_s'], 1)[0] * 100 if len(late) > 5 else np.nan
    # 초기 구간은 용량이 오히려 소폭 증가하기도 해 기울기가 0/음수일 수 있음 → 가속 비율은 중기 대비로 계산

    # Knee : 시작점-끝점 직선에서 가장 멀리 위에 있는 점 (정규화 곡선 기준, Kneedle 방식)
    x = (c['cycle'] - c['cycle'].iloc[0]) / (c['cycle'].iloc[-1] - c['cycle'].iloc[0])
    y = (c['QD_s'] - c['QD_s'].iloc[-1]) / (c['QD_s'].iloc[0] - c['QD_s'].iloc[-1])
    k = int(np.argmax(y.values + x.values - 1))
    return pd.Series({
        'slope_early': slope_early, 'slope_mid': slope_mid, 'slope_late': slope_late,
        'accel_ratio(late/mid)': slope_late / slope_mid,
        'knee_cycle': c['cycle'].iloc[k], 'knee_QD': c['QD_s'].iloc[k], 'knee_ratio': c['cycle'].iloc[k] / life,
        'dQD_2_100': (early['QD_s'].iloc[0] - early['QD_s'].iloc[-1]) if len(early) else np.nan,
    })

q2 = curves.groupby('cell_id').apply(fade_features, include_groups=False).reset_index()
q2 = q2.merge(ci[['cell_id', 'batch', 'structure', 'policy', 'cycle_life']], on='cell_id')
display(q2.groupby('batch')[['slope_early', 'slope_mid', 'slope_late', 'accel_ratio(late/mid)', 'knee_ratio', 'knee_QD', 'dQD_2_100']]
        .median().round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for b, g in q2.groupby('batch'):
    axes[0].scatter(g['knee_cycle'], g['cycle_life'], color=BATCH_COLOR[b], alpha=0.7, label=BATCH_LABEL[b])
    axes[1].hist(g['knee_ratio'], bins=np.linspace(0.3, 1, 15), alpha=0.5, color=BATCH_COLOR[b], label=BATCH_LABEL[b])
    axes[2].scatter(g['slope_early'], g['cycle_life'], color=BATCH_COLOR[b], alpha=0.7, label=BATCH_LABEL[b])
lim = [0, q2['cycle_life'].max() * 1.05]
axes[0].plot(lim, lim, 'k:', lw=1)
r = stats.pearsonr(q2['knee_cycle'], q2['cycle_life'])[0]
axes[0].set(xlabel='Knee cycle', ylabel='Cycle Life', title=f'Knee cycle vs Life (r = {r:.2f})')
axes[1].set(xlabel='Knee cycle / Cycle Life', ylabel='Number of cells', title='Where does the knee occur?')
axes[2].set(xlabel='Early fade rate, cycles 2-100 (Ah / 100 cycles)', ylabel='Cycle Life',
            title='Early fade rate vs Life')
axes[0].legend(fontsize=9)
plt.suptitle('Q2. Fade Rate & Knee Point', fontweight='bold')
plt.tight_layout(); savefig('Q2_knee_fade_rate'); plt.show()

### Q2-5. 초기 100사이클 확대 : 용량 값만으로 구분 가능한가?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)
for ax, (b, g) in zip(axes, curves[curves['cycle'] <= 100].groupby('batch')):
    for cid, c in g.groupby('cell_id'):
        ax.plot(c['cycle'], c['QD_s'], color=cmap(norm(c['cycle_life'].iloc[0])), lw=0.8)
    ax.set(title=BATCH_LABEL[b], xlabel='Cycle (2-100)')
axes[0].set_ylabel('QD (Ah)')
fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, label='Cycle Life', pad=0.01)
plt.suptitle('Q2. First 100 Cycles : capacity alone cannot separate long/short life', fontweight='bold')
savefig('Q2_first100_cycles'); plt.show()

print('[cycles 2-100 용량 감소량 vs log10(수명) 상관 (배치별)]')
for b, g in q2.groupby('batch'):
    r = stats.spearmanr(g['dQD_2_100'], np.log10(g['cycle_life']))[0]
    print(f'  {b}: Spearman r = {r:.3f}')

### Q2 정리

**[핵심 발견]**
- 열화 형태 : 모든 배치에서 "완만한 구간 → 급격한 하락"의 비선형 열화.
  정규화 곡선 중앙값 기준 수명의 약 60~70%까지는 용량 손실이 3~5% 수준이다가, 이후 0.80~0.82까지 급락 (직선 기준선과 큰 차이)
- 열화 속도 : 중기(수명 40~60%) 대비 말기(마지막 10%) 기울기가 약 11~12배 (Batch 1 10.9배, Batch 2 10.6배, Batch 3 12.5배)
- Knee point : 수명의 약 69~76% 지점(중앙값 Batch 1 0.71 / Batch 2 0.69 / Batch 3 0.76)에서 발생, knee cycle과 수명의 상관 r = 0.99
  - (주의) knee는 EOL까지의 전체 곡선으로 계산한 값이라 예측 피처로 사용 불가. "수명 예측 ≒ knee 시점 예측"이라는 해석에만 활용
- 초기 100사이클 :
  - 용량 변화가 거의 없거나 오히려 증가 (cycles 2-100 용량 변화 중앙값 -0.001~-0.002Ah, 즉 소폭 증가)
  - 수명별 곡선이 섞여 있어 용량 값만으로 장/단수명 구분 불가
  - 초기 용량 감소량과 log10(수명)의 상관 : Batch 1 r = -0.46, Batch 2 r = 0.17, Batch 3 r = 0.13 → 학습 배치에서 보이는 상관이 테스트 배치에서 사라짐
- 배치별 초기 용량 차이 : Batch 2 약 1.08~1.11Ah(측정 노이즈, 계단형 점프 존재), Batch 3 약 1.04~1.08Ah로 낮음
  → 용량 절대값 피처는 배치마다 기준이 달라 일반화 어려움

**[시사점 → 모델링 반영]**
- 초기 100사이클에는 knee 이전의 완만한 구간만 보이므로, 용량 곡선 자체로는 미래의 급락 시점을 알 수 없음
  → 용량 "합계"가 아니라 전압 구간별 용량 변화(ΔQ(V))처럼 내부 열화를 더 민감하게 보는 피처 필요 (Q3에서 확인)
- 용량 기반 피처(초기 감소량, 기울기)는 학습 배치에서만 상관 → 보조 피처 후보로만 두고 Q5에서 재검토
- 용량 절대값 대신 차이·비율 형태의 피처 사용 (배치 간 초기 용량 차이 제거)
- Batch 2의 초기 구간 계단형 점프 → ΔQ(V) 계산 사이클(10, 100) 근처에 점프가 있는지 Q3에서 확인

---
## Q3. ΔQ(V) 곡선 : 초기 100사이클에서 수명 신호를 찾을 수 있는가?
- ΔQ(V) = Q_100(V) − Q_10(V) : 같은 전압에서 사이클 100과 사이클 10의 방전 용량 차이 (Severson 2019 핵심 피처)
- 사이클 번호는 `summary['cycle']` 기준 (사이클 10 = cycle 값 10)
- ΔQ(V) 곡선의 통계량(min, mean, var, skew, kurtosis)과 log10(수명)의 관계
- **Batch 1에서 찾은 관계가 Batch 2·3에서도 유지되는가?** (일반화 확인)

In [ ]:
cells = data['cells']
CYC_A, CYC_B = 10, 100

def get_qdlin(c, n):
    q = c['Qdlin']
    if n in q:
        return q[n].astype(float), n
    k = min(q, key=lambda x: abs(x - n))      # 해당 사이클이 없으면 가장 가까운 사이클
    return q[k].astype(float), k

rows, dq_curves = [], {}
for cid in ci['cell_id']:
    c = cells[cid]
    qa, na = get_qdlin(c, CYC_A)
    qb, nb_ = get_qdlin(c, CYC_B)
    dq = qb - qa
    dq_curves[cid] = dq
    rows.append({'cell_id': cid, 'cyc_a': na, 'cyc_b': nb_,
                 'dQ_min': dq.min(), 'dQ_mean': dq.mean(), 'dQ_var': dq.var(),
                 'dQ_skew': stats.skew(dq), 'dQ_kurt': stats.kurtosis(dq)})

q3 = pd.DataFrame(rows).merge(ci[['cell_id', 'batch', 'structure', 'policy', 'cycle_life']], on='cell_id')
q3['log_var'] = np.log10(q3['dQ_var'])
q3['log_abs_min'] = np.log10(q3['dQ_min'].abs())
q3['log_life'] = np.log10(q3['cycle_life'])

print('사이클 10/100 대체 사용 셀:', q3[(q3.cyc_a != CYC_A) | (q3.cyc_b != CYC_B)]['cell_id'].tolist())
display(q3.groupby('batch')[['dQ_min', 'dQ_mean', 'dQ_var', 'log_var']].median())

### Q3-1. ΔQ(V) 곡선 (배치별, 색 = 수명)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
for ax, (b, g) in zip(axes, q3.groupby('batch')):
    for _, r in g.iterrows():
        c = cells[r['cell_id']]
        V = c['Vdlin'] if c['Vdlin'] is not None else np.linspace(3.6, 2.0, len(dq_curves[r['cell_id']]))
        ax.plot(dq_curves[r['cell_id']], V, color=cmap(norm(r['cycle_life'])), lw=0.8)
    ax.axvline(0, color='gray', lw=0.8)
    ax.set(title=BATCH_LABEL[b], xlabel='ΔQ = Q100 − Q10 (Ah)')
axes[0].set_ylabel('Voltage (V)')
fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, label='Cycle Life', pad=0.01)
plt.suptitle('Q3. ΔQ(V) Curves (color = cycle life)', fontweight='bold')
savefig('Q3_deltaQ_curves'); plt.show()

### Q3-2. log10(var ΔQ) vs log10(수명) : Batch 1 관계가 다른 배치에도 유지되는가?
Batch 1으로 단순 선형식(log 수명 = a × log var + b)을 맞추고, 같은 직선을 Batch 2·3에 그대로 적용해 본다.
(모델링이 아니라 "피처가 배치를 넘어 일반화되는가"를 보는 EDA 확인용)

In [ ]:
b1 = q3[q3.batch == 'b1']
a, b0 = np.polyfit(b1['log_var'], b1['log_life'], 1)
q3['pred_life_b1line'] = 10 ** (a * q3['log_var'] + b0)
q3['ape'] = (q3['pred_life_b1line'] - q3['cycle_life']).abs() / q3['cycle_life'] * 100

NOISY_B3 = ['b3c2', 'b3c37', 'b3c42', 'b3c43']     # 원논문 노이즈 채널

fig, ax = plt.subplots(figsize=(10, 6.5))
for (bt, st), g in q3.groupby(['batch', 'structure']):
    ax.scatter(g['log_var'], g['log_life'], color=BATCH_COLOR[bt], marker='o' if st == 'old' else '^',
               alpha=0.75, s=45, label=f'{BATCH_LABEL[bt]} / {st}')
nz = q3[q3.cell_id.isin(NOISY_B3)]
ax.scatter(nz['log_var'], nz['log_life'], facecolors='none', edgecolors='red', s=160, lw=1.5, label='paper-excluded b3 (noisy)')
xs = np.linspace(q3['log_var'].min(), q3['log_var'].max(), 50)
ax.plot(xs, a * xs + b0, 'k--', lw=1.5, label=f'Batch 1 fit: y = {a:.2f}x + {b0:.2f}')
ax.set(xlabel='log10(var(ΔQ))', ylabel='log10(Cycle Life)', title='Q3. log10 var(ΔQ) vs log10 Cycle Life')
ax.legend(fontsize=9)
plt.tight_layout(); savefig('Q3_logvar_vs_life'); plt.show()

summary_rows = []
for bt, g in q3.groupby('batch'):
    summary_rows.append({'batch': bt, 'n': len(g),
                         'Pearson r (log_var, log_life)': stats.pearsonr(g['log_var'], g['log_life'])[0],
                         'MAPE with Batch 1 line (%)': g['ape'].mean()})
for st, g in q3[q3.batch == 'b2'].groupby('structure'):
    summary_rows.append({'batch': f'b2-{st}', 'n': len(g),
                         'Pearson r (log_var, log_life)': stats.pearsonr(g['log_var'], g['log_life'])[0] if len(g) > 2 else np.nan,
                         'MAPE with Batch 1 line (%)': g['ape'].mean()})
display(pd.DataFrame(summary_rows).round(3))
print('[원논문 노이즈 셀 (Batch 3)]')
display(q3[q3.cell_id.isin(NOISY_B3)][['cell_id', 'cycle_life', 'log_var', 'pred_life_b1line', 'ape']].round(2))

### Q3-3. ΔQ(V) 통계량별 log10(수명) 상관 (배치별)

In [ ]:
feat = ['log_var', 'log_abs_min', 'dQ_mean', 'dQ_skew', 'dQ_kurt']
corr_tbl = pd.DataFrame({bt: [stats.pearsonr(g[f], g['log_life'])[0] for f in feat]
                         for bt, g in q3.groupby('batch')}, index=feat).round(3)
display(corr_tbl)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), sharey=True)
for ax, f in zip(axes, ['log_abs_min', 'dQ_mean', 'dQ_skew', 'dQ_kurt']):
    for bt, g in q3.groupby('batch'):
        ax.scatter(g[f], g['log_life'], color=BATCH_COLOR[bt], alpha=0.7, s=25, label=BATCH_LABEL[bt])
    ax.set(xlabel=f)
axes[0].set_ylabel('log10(Cycle Life)'); axes[0].legend(fontsize=8)
plt.suptitle('Q3. Other ΔQ(V) statistics vs log10 Cycle Life', fontweight='bold')
plt.tight_layout(); savefig('Q3_deltaQ_stats'); plt.show()

### Q3 정리

**[핵심 발견]**
- ΔQ(V) 곡선 형태 : 수명이 짧을수록 곡선이 왼쪽(음수)으로 크게 벌어짐. 변화는 주로 3.0~3.2V 방전 평탄 구간에 집중
  - 배치별 var(ΔQ) 중앙값 : Batch 2(10^-3.49) > Batch 1(10^-3.79) > Batch 3(10^-4.18) → 배치별 수명 순서(472 < 772 < 1006)와 일치
  - Batch 3 장수명 셀 일부는 3.0~3.1V에서 ΔQ가 양수(용량 증가), 일부 곡선에 수직 스파이크(측정 노이즈) 존재
- log10(var ΔQ)와 log10(수명)의 상관 : Batch 1 r = -0.844, Batch 2 r = -0.918, Batch 3 r = -0.762
  → **세 배치 모두 강한 음의 선형관계** (Q2의 용량 감소량은 테스트 배치에서 상관이 사라졌던 것과 대조)
- 일반화 (Batch 1 직선 y = -0.30x + 1.76 적용) :
  - Batch 2 newstructure, Batch 3 대부분은 직선 근처 → 관계가 그대로 유지
  - Batch 2 기존 구조(old) 셀은 직선보다 일관되게 아래 → 같은 var(ΔQ)에서 실제 수명이 더 짧음 = 과대예측(체계적 오프셋)
  - Batch 2 old 셀의 log var(-3.7 ~ -3.1)는 Batch 1 범위(-4.2 ~ -3.35) 바깥까지 분포 → x축에서도 외삽 발생
- 다른 통계량 : log|min ΔQ|(r = -0.74 ~ -0.90), mean ΔQ(r = 0.72 ~ 0.80)도 세 배치에서 일관.
  skew(-0.43 ~ 0.19), kurtosis(0.02 ~ 0.65)는 배치마다 부호·크기가 바뀌어 불안정
- 원논문 노이즈 셀(b3c2, c37, c42, c43) : 2개는 var(ΔQ)가 매우 작아 학습 범위 밖(왼쪽 끝), 직선 대비 오차 방향이 셀마다 다름

**[시사점 → 모델링 반영]**
- 핵심 피처 : log10(var ΔQ) 채택 (세 배치 모두 |r| ≥ 0.76, 배치를 넘어 일반화되는 유일한 강한 신호)
- 보조 피처 후보 : log|min ΔQ|, mean ΔQ (log var와 상관이 높을 가능성 → Q5에서 다중공선성 확인)
- 제외 : skew, kurtosis (배치 간 불안정)
- 타깃 : log10(cycle_life) 확정 (log-log 선형관계, Q1 skew 개선과 일치)
- 모델 : 단일 피처로도 선형관계가 강함 → 선형 모델(ElasticNet/Ridge) 근거 강화.
  log var 축에서도 외삽이 필요하므로 트리 계열은 Batch 2 old 셀처럼 범위 밖 셀에서 불리할 것으로 예상
- 위험 요인 : Batch 2 old 셀의 체계적 과대예측 → 오류 분석에서 구조별로 분리 보고
- Batch 3 노이즈 셀 : EDA에서는 유지, DAY2 Batch 3 평가 시 포함/제외 두 경우를 모두 보고

---
## Q4. 충전 조건(C-rate)과 수명 : 고속충전일수록 수명이 짧은가?
- 정책 파싱 : `C1(SOC1%)-C2` → 1단계 전류 C1, 전환 SOC, 2단계 전류 C2
- 0→80% 충전시간(계산) = 60 × (SOC1/C1 + (80 − SOC1)/C2) / 100 (분), 평균 C-rate = 0.8 / (충전시간/60)
- 정책별 평균 수명, C-rate·온도와 수명의 관계, 배치별 일관성 확인

In [ ]:
q4 = ci[['cell_id', 'batch', 'structure', 'policy', 'cycle_life']].copy()
q4 = pd.concat([q4, q4['policy'].apply(parse_policy)], axis=1)
q4['t80_calc'] = 60 * (q4['switch_SOC'] / q4['C1'] + (80 - q4['switch_SOC']) / q4['C2']) / 100
q4['C_avg'] = 0.8 / (q4['t80_calc'] / 60)
q4 = q4.merge(early[['early_chargetime', 'early_Tmax', 'early_Tavg']], left_on='cell_id', right_index=True)
q4['log_life'] = np.log10(q4['cycle_life'])

print('정책 파싱 실패 셀:', q4[q4['C1'].isna()][['cell_id', 'policy']].values.tolist())
print(f"계산 충전시간 vs 측정 충전시간 상관 : r = {q4[['t80_calc', 'early_chargetime']].dropna().corr().iloc[0, 1]:.3f}")
display(q4.groupby('batch')[['C1', 'C2', 'switch_SOC', 't80_calc', 'C_avg']].agg(['min', 'median', 'max']).round(2))

### Q4-1. 정책별 평균 수명

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 9))
for ax, (bt, g) in zip(axes, q4.groupby('batch')):
    p = g.groupby('policy')['cycle_life'].agg(['mean', 'count']).sort_values('mean')
    ax.barh(range(len(p)), p['mean'], color=BATCH_COLOR[bt], alpha=0.6)
    for i, pol in enumerate(p.index):
        y = g.loc[g.policy == pol, 'cycle_life']
        ax.scatter(y, [i] * len(y), color='k', s=12, zorder=3)
    ax.set_yticks(range(len(p)))
    ax.set_yticklabels([f"{pol.replace('-newstructure', '*')} (n={n})" for pol, n in zip(p.index, p['count'])], fontsize=8)
    ax.set(title=BATCH_LABEL[bt], xlabel='Cycle Life (bar = mean, dot = cell)')
plt.suptitle('Q4. Cycle Life by Charging Policy  (* = newstructure)', fontweight='bold')
plt.tight_layout(); savefig('Q4_policy_life'); plt.show()

### Q4-2. C-rate / 온도와 수명

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4.8), sharey=True)
for ax, (col, lab) in zip(axes, [('C1', 'Step-1 C-rate (C1)'), ('C2', 'Step-2 C-rate (C2)'),
                                 ('C_avg', 'Average C-rate to 80% SOC'), ('early_Tmax', 'Mean Tmax, cycles 2-100 (C)')]):
    for (bt, st), g in q4.groupby(['batch', 'structure']):
        ax.scatter(g[col], g['cycle_life'], color=BATCH_COLOR[bt], marker='o' if st == 'old' else '^',
                   alpha=0.7, s=30, label=f'{BATCH_LABEL[bt]} / {st}')
    ax.set(xlabel=lab)
axes[0].set_ylabel('Cycle Life'); axes[0].legend(fontsize=8)
plt.suptitle('Q4. Charging Condition & Temperature vs Cycle Life', fontweight='bold')
plt.tight_layout(); savefig('Q4_crate_temp_life'); plt.show()

cols = ['C1', 'switch_SOC', 'C2', 'C_avg', 't80_calc', 'early_chargetime', 'early_Tmax', 'early_Tavg']
q4_corr = pd.DataFrame({bt: [stats.spearmanr(g[c], g['log_life'], nan_policy='omit')[0] for c in cols]
                        for bt, g in q4.groupby('batch')}, index=cols).round(3)
print('[조건 변수 vs log10(수명) Spearman 상관, 배치별]')
display(q4_corr)

### Q4 정리

**[핵심 발견]**
- 정책별 수명 : Batch 1은 5.4C(80%)-5.4C가 가장 짧음(약 550), Batch 2 기존 구조는 정책과 무관하게 약 400~510,
  Batch 3은 4.8C(80%)-4.8C가 가장 길고(평균 약 1570) 3.7C(31%)-5.9C가 가장 짧음(약 660)
- C-rate : C1(-0.24 / 0.06 / -0.23), C2(-0.16 / -0.12 / 0.02)는 약하고, 평균 C-rate(-0.43 / 0.35 / -0.13)는 배치마다 부호가 뒤집힘
- 충전시간 : Batch 2·3은 평균 C-rate가 전부 4.8C(0→80% 10분)로 같음 → 10분 충전 설계 확인
  - Batch 2 충전시간 상관(-0.76)은 newstructure 10.04분 vs 기존 구조 약 10.17분 차이에서 생긴 구조 효과 (가짜 상관)
- 온도 : 평균 최고온도와 수명은 세 배치 모두 관계 없음 (ρ = -0.14 ~ -0.04)
- 같은 정책 셀끼리 수명이 비슷함

**[시사점 → 모델링 반영]**
- 충전 조건 변수(C-rate, 충전시간)와 온도는 피처에서 제외 (배치 간 일관성 없음, 구조 효과에 오염)
- 정책보다 셀 구조·배치 영향이 큼 → 조건이 아닌 셀의 열화 상태(ΔQ(V))로 예측해야 하는 근거
- 검증은 정책 단위 GroupKFold (같은 정책 셀이 학습·검증에 나뉘는 누수 방지)

---
## Q5. 초기 사이클 피처 간 상관관계와 다중공선성
- 원논문 피처 체계(ΔQ(V), 용량 곡선, 충전시간, 온도, 내부저항)를 초기 100사이클로 계산
- 정제 데이터 기준 (사이클 0 제거, 이상치 행 제외)
- Batch 1(학습) 상관 히트맵 → 다중공선성 쌍 확인 / 배치별 log 수명 상관 → 일반화되는 피처 선별

In [ ]:
def early_cycle_features(g):
    g = g.sort_values('cycle')
    e = g[(g['cycle'] >= 2) & (g['cycle'] <= 100)]
    slope, intercept = np.polyfit(e['cycle'], e['QD'], 1)
    ir = e.loc[e['IR'] > 0, 'IR']
    return pd.Series({
        'QD_2': e['QD'].iloc[0],
        'QD_max_minus_2': e['QD'].max() - e['QD'].iloc[0],
        'QD_100_minus_2': e['QD'].iloc[-1] - e['QD'].iloc[0],
        'fade_slope_2_100': slope,
        'fade_intercept_2_100': intercept,
        'chargetime_2_6': g.loc[(g['cycle'] >= 2) & (g['cycle'] <= 6), 'chargetime'].mean(),
        'Tavg_mean': e['Tavg'].mean(), 'Tmax_mean': e['Tmax'].mean(), 'Tmin_mean': e['Tmin'].mean(),
        'Tmax_max': e['Tmax'].max(),
        'IR_2': ir.iloc[0] if len(ir) else np.nan,
        'IR_min': ir.min() if len(ir) else np.nan,
        'IR_100_minus_2': (ir.iloc[-1] - ir.iloc[0]) if len(ir) else np.nan,
    })

f_cycle = summary.groupby('cell_id').apply(early_cycle_features, include_groups=False)
feat_df = (q3[['cell_id', 'batch', 'structure', 'policy', 'cycle_life', 'log_life',
               'log_var', 'log_abs_min', 'dQ_mean', 'dQ_skew', 'dQ_kurt']]
           .merge(f_cycle, left_on='cell_id', right_index=True))
FEATS = ['log_var', 'log_abs_min', 'dQ_mean', 'dQ_skew', 'dQ_kurt',
         'QD_2', 'QD_max_minus_2', 'QD_100_minus_2', 'fade_slope_2_100', 'fade_intercept_2_100',
         'chargetime_2_6', 'Tavg_mean', 'Tmax_mean', 'Tmin_mean', 'Tmax_max', 'IR_2', 'IR_min', 'IR_100_minus_2']

out = os.path.join(ROOT, 'processed', 'features_day1.csv')
feat_df.to_csv(out, index=False)
print(f'피처 테이블 저장 : {out}  shape = {feat_df.shape}')
print('결측 :', feat_df[FEATS].isna().sum()[lambda s: s > 0].to_dict())

### Q5-1. Batch 1(학습) 피처 상관 히트맵

In [ ]:
b1f = feat_df[feat_df.batch == 'b1']
cm = b1f[FEATS + ['log_life']].corr()
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(cm))); ax.set_xticklabels(cm.columns, rotation=60, ha='right', fontsize=9)
ax.set_yticks(range(len(cm))); ax.set_yticklabels(cm.index, fontsize=9)
for i in range(len(cm)):
    for j in range(len(cm)):
        ax.text(j, i, f'{cm.iloc[i, j]:.2f}', ha='center', va='center', fontsize=6.5,
                color='white' if abs(cm.iloc[i, j]) > 0.6 else 'black')
ax.grid(False); fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Q5. Feature Correlation (Batch 1, cycles 2-100, cleaned)', fontweight='bold')
plt.tight_layout(); savefig('Q5_corr_heatmap_b1'); plt.show()

pairs = [(a, b, cm.loc[a, b]) for i, a in enumerate(FEATS) for b in FEATS[i + 1:] if abs(cm.loc[a, b]) >= 0.9]
print('[|r| >= 0.9 피처 쌍 (Batch 1)]')
display(pd.DataFrame(pairs, columns=['feature_a', 'feature_b', 'r']).sort_values('r', key=abs, ascending=False).round(3))

### Q5-2. 피처별 log10(수명) 상관 : 배치 간 일관성

In [ ]:
rows = []
for f in FEATS:
    r = {bt: stats.pearsonr(g[f], g['log_life'])[0] if g[f].notna().sum() > 2 else np.nan
         for bt, g in feat_df.groupby('batch')}
    vals = np.array(list(r.values()))
    r['일관성'] = '일관' if (np.all(np.sign(vals) == np.sign(vals[0])) and np.all(np.abs(vals) >= 0.3)) else ''
    rows.append(pd.Series(r, name=f))
q5_corr = pd.DataFrame(rows)
q5_corr['|r| min'] = q5_corr[['b1', 'b2', 'b3']].abs().min(axis=1)
q5_corr = q5_corr.sort_values('|r| min', ascending=False)
display(q5_corr.round(3))

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(q5_corr)); w = 0.27
for k, bt in enumerate(['b1', 'b2', 'b3']):
    ax.bar(x + (k - 1) * w, q5_corr[bt], w, color=BATCH_COLOR[bt], label=BATCH_LABEL[bt])
ax.axhline(0, color='k', lw=0.8)
for y in (0.3, -0.3):
    ax.axhline(y, color='gray', ls=':', lw=1)
ax.set_xticks(x); ax.set_xticklabels(q5_corr.index, rotation=45, ha='right')
ax.set(ylabel='Pearson r with log10(Cycle Life)', title='Q5. Feature-Life Correlation by Batch (sorted by min |r|)')
ax.legend()
plt.tight_layout(); savefig('Q5_feature_life_corr_by_batch'); plt.show()

### Q5-3. 다중공선성 (VIF, Batch 1) : 일관 피처 후보 대상

In [ ]:
def vif_table(X):
    X = (X - X.mean()) / X.std()
    out = {}
    for c in X.columns:
        y, others = X[c].values, X.drop(columns=c).values
        A = np.column_stack([np.ones(len(y)), others])
        coef, *_ = np.linalg.lstsq(A, y, rcond=None)
        r2 = 1 - ((y - A @ coef) ** 2).sum() / ((y - y.mean()) ** 2).sum()
        out[c] = 1 / (1 - r2) if r2 < 1 else np.inf
    return pd.Series(out, name='VIF').sort_values(ascending=False)

cand = q5_corr.index[q5_corr['일관성'] == '일관'].tolist()
print('배치 일관 피처 후보 :', cand)
if len(cand) >= 2:
    display(vif_table(b1f[cand].dropna()).round(2).to_frame())

### Q5 정리

**[핵심 발견]**
- 다중공선성 쌍 (Batch 1, |r| ≥ 0.9) : QD_2-fade_intercept 0.998, log_var-log_abs_min 0.996, QD_100_minus_2-fade_slope 0.995,
  log_abs_min-dQ_mean -0.977, Tmax_mean-Tmax_max 0.976, log_var-dQ_mean -0.971, Tavg_mean-Tmax_mean 0.953
- 배치 간 일관 피처 : log_var(-0.84 / -0.92 / -0.76), log_abs_min(-0.82 / -0.90 / -0.74), dQ_mean(0.80 / 0.79 / 0.72) 3개뿐
- 학습 배치에서만 상관 : chargetime_2_6(0.58 → Batch 2 -0.94), fade_slope_2_100(0.55 → Batch 2 -0.36) → 부호 뒤집힘
- 온도는 관계 없음, 내부저항은 Batch 2에서 결측·상수로 상관 계산 불가
- VIF (일관 피처 3개) : log_abs_min 183, log_var 145, dQ_mean 23 → 사실상 같은 정보

**[시사점 → 모델링 반영]**
- 최종 피처 세트
  - A. Variance : log_var (베이스라인)
  - B. ΔQ : log_var, log_abs_min, dQ_mean (메인, ElasticNet으로 공선성 대응)
  - C. Full : Q5 피처 전체(IR 제외) (대조군, 학습 배치 전용 상관 피처의 일반화 실패 확인)
- Batch 1 상관 순위만으로 피처를 고르면 테스트에서 실패 → 배치 간 일관성을 피처 선택 기준으로 사용